In [0]:
# 01 - Importar funções e carregar a Gold de Machine Learning

from pyspark.sql import functions as F

df_ml = spark.table(
    "workspace.projeto_integrador.gold_base_ml"
)

print("Total de registros:", df_ml.count())

display(
    df_ml
    .orderBy("codigo_ibge", "ano")
    .limit(20)
)

In [0]:
# 02 - Criar variável alvo e novas features numéricas

df_features = (
    df_ml

    # Variável que queremos prever
    .withColumn(
        "target",
        F.col("valor_indicador_municipio").cast("double")
    )

    # Transformação logarítmica da variável alvo
    .withColumn(
        "target_log",
        F.log(
            F.col("valor_indicador_municipio") + F.lit(1.0)
        )
    )

    # Versões logarítmicas dos valores históricos
    .withColumn(
        "log_valor_ano_anterior",
        F.log(
            F.col("valor_ano_anterior") + F.lit(1.0)
        )
    )

    .withColumn(
        "log_valor_2_anos_antes",
        F.log(
            F.col("valor_2_anos_antes") + F.lit(1.0)
        )
    )

    .withColumn(
        "log_valor_3_anos_antes",
        F.log(
            F.col("valor_3_anos_antes") + F.lit(1.0)
        )
    )

    .withColumn(
        "log_media_3_anos",
        F.log(
            F.col("media_3_anos_anteriores") + F.lit(1.0)
        )
    )

    # Variação percentual entre os dois últimos anos disponíveis
    .withColumn(
        "variacao_percentual_historica",
        F.when(
            F.col("valor_2_anos_antes") > 0,
            (
                (
                    F.col("valor_ano_anterior") -
                    F.col("valor_2_anos_antes")
                )
                / F.col("valor_2_anos_antes")
            ) * 100
        ).otherwise(0.0)
    )

    # Relação entre o último ano e a média dos três anteriores
    .withColumn(
        "razao_ultimo_media_3_anos",
        F.when(
            F.col("media_3_anos_anteriores") > 0,
            F.col("valor_ano_anterior") /
            F.col("media_3_anos_anteriores")
        ).otherwise(0.0)
    )
)

display(
    df_features
    .orderBy("codigo_ibge", "ano")
    .limit(20)
)

In [0]:
# 03 - Criar divisão temporal dos dados

df_features = (
    df_features
    .withColumn(
        "conjunto",
        F.when(
            F.col("ano") <= 2023,
            "treino"
        )
        .when(
            F.col("ano") == 2024,
            "validacao"
        )
        .when(
            F.col("ano") == 2025,
            "teste"
        )
    )
)

distribuicao_conjuntos = (
    df_features
    .groupBy("conjunto")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("codigo_ibge").alias("municipios"),
        F.min("ano").alias("ano_inicial"),
        F.max("ano").alias("ano_final")
    )
    .orderBy("ano_inicial")
)

display(distribuicao_conjuntos)

In [0]:
# 04 - Separar treino, validação e teste

df_treino = (
    df_features
    .filter(F.col("conjunto") == "treino")
)

df_validacao = (
    df_features
    .filter(F.col("conjunto") == "validacao")
)

df_teste = (
    df_features
    .filter(F.col("conjunto") == "teste")
)

print("Registros de treino:", df_treino.count())
print("Registros de validação:", df_validacao.count())
print("Registros de teste:", df_teste.count())

In [0]:
# 05 - Verificar valores zero em cada conjunto

analise_zeros = (
    df_features
    .groupBy("conjunto")
    .agg(
        F.count("*").alias("total_registros"),

        F.sum(
            F.when(
                F.col("target") == 0,
                1
            ).otherwise(0)
        ).alias("quantidade_zeros"),

        F.sum(
            F.when(
                F.col("target") > 0,
                1
            ).otherwise(0)
        ).alias("quantidade_maior_zero")
    )
    .withColumn(
        "percentual_zeros",
        F.round(
            (
                F.col("quantidade_zeros") /
                F.col("total_registros")
            ) * 100,
            2
        )
    )
)

display(analise_zeros)

In [0]:
# 06 - Estatísticas do target por conjunto

estatisticas_target = (
    df_features
    .groupBy("conjunto")
    .agg(
        F.count("*").alias("registros"),

        F.round(
            F.avg("target"),
            2
        ).alias("media"),

        F.expr(
            "percentile_approx(target, 0.5)"
        ).alias("mediana"),

        F.min("target").alias("minimo"),

        F.max("target").alias("maximo")
    )
    .orderBy("conjunto")
)

display(estatisticas_target)

In [0]:
# 07 - Verificar valores nulos nas features de Machine Learning

colunas_ml = [
    "ano",
    "valor_ano_anterior",
    "valor_2_anos_antes",
    "valor_3_anos_antes",
    "media_3_anos_anteriores",
    "diferenca_ultimo_ano",
    "variacao_percentual_historica",
    "razao_ultimo_media_3_anos",
    "log_valor_ano_anterior",
    "log_valor_2_anos_antes",
    "log_valor_3_anos_antes",
    "log_media_3_anos",
    "target",
    "target_log"
]

expressoes_nulos = [
    F.sum(
        F.when(
            F.col(coluna).isNull(),
            1
        ).otherwise(0)
    ).alias(coluna)
    for coluna in colunas_ml
]

nulos = df_features.select(*expressoes_nulos)

display(nulos)

In [0]:
# 08 - Selecionar colunas relevantes para modelagem

df_modelagem = (
    df_features
    .select(
        # Identificação
        "codigo_ibge",
        "municipio",
        "uf",
        "estado",
        "regiao_brasil",

        # Tempo
        "ano",

        # Features históricas
        "valor_ano_anterior",
        "valor_2_anos_antes",
        "valor_3_anos_antes",
        "media_3_anos_anteriores",
        "diferenca_ultimo_ano",
        "variacao_percentual_historica",
        "razao_ultimo_media_3_anos",

        # Features logarítmicas
        "log_valor_ano_anterior",
        "log_valor_2_anos_antes",
        "log_valor_3_anos_antes",
        "log_media_3_anos",

        # Alvos
        "target",
        "target_log",

        # Separação temporal
        "conjunto"
    )
)

display(
    df_modelagem
    .orderBy("codigo_ibge", "ano")
    .limit(20)
)

In [0]:
# 09 - Salvar tabela de features para Machine Learning

df_modelagem.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(